In [ ]:
# install ffmpeg Go to https://ffmpeg.org/download.html
# conda install -c conda-forge ffmpeg

import pandas as pd
import bar_chart_race as bcr
from raceplotly.plots import barplot
import matplotlib.pyplot as plt

### race plot documentation : https://www.dexplo.org/bar_chart_race/tutorial/

# Import data

In [2]:
# import macro data  (duration 1m35)

path = 'F:/Divers/WDI_EXCEL_2025_01_28/WDIEXCEL.xlsx'
sheet  = 'Data'
data_df = pd.read_excel(path, sheet_name=sheet )

data_df.head(5)

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,17.488497,18.001597,18.558234,19.043572,19.586457,20.192064,20.828814,21.372164,22.100884,NaN
1,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,6.811504,7.096003,7.406706,7.666648,8.020952,8.403358,8.718306,9.097176,9.473374,NaN
2,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,38.152090,38.488233,38.779953,39.068462,39.445526,39.818645,40.276374,40.687817,41.211606,NaN
3,Africa Eastern and Southern,AFE,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,31.871956,33.922276,38.859598,40.223744,43.035073,44.390861,46.282371,48.127211,48.742043,NaN
4,Africa Eastern and Southern,AFE,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,17.672943,16.527554,24.627753,25.432092,27.061929,29.154282,31.022083,32.809138,33.760782,NaN


In [3]:
# import countries selection for Africa

path = "Resources/coutries_africa.csv"
countries_select_df = pd.read_csv(path, sep=';' )
countries_select_df.head(5)

,Country Name,Country Code
0,Algeria,DZA
1,Angola,AGO
2,Benin,BEN
3,Botswana,BWA
4,Burkina Faso,BFA


In [4]:
# import indicators selection

path = "Resources/indicators_selection - bar race.xlsx"
indicators_df = pd.read_excel(path )

## filter the selected indicators
x = (indicators_df['select'] == 'y')
indicators_df = indicators_df[x]
## drop column 'select'
indicators_df = indicators_df.drop(columns = {'select'})
indicators_df.head(5)

,Indicator Name,Indicator Code,indicator_new,Category
6,Arable land (hectares per person),AG.LND.ARBL.HA.PC,Arable land,agriculture
19,Market capitalization of listed domestic compa...,CM.MKT.LCAP.GD.ZS,Market capitalization of listed domestic compa...,business
48,Access to electricity (% of population),EG.ELC.ACCS.ZS,Access to electricity,energy
49,Electric power consumption (kWh per capita),EG.USE.ELEC.KH.PC,Electric power consumption,energy
61,"Birth rate, crude (per 1,000 people)",SP.DYN.CBRT.IN,Birth rate,population


### create list of indicators
these dictionaries will be used in the  chart and to name the files
so they cannot have the same structure

In [5]:
indicators_df_new = indicators_df[['Indicator Name', 'indicator_new']]
indicators_list = indicators_df_new.to_dict(orient='records')
indicators_list

[{'Indicator Name': 'Arable land (hectares per person)',
  'indicator_new': 'Arable land'},
 {'Indicator Name': 'Market capitalization of listed domestic companies (% of GDP)',
  'indicator_new': 'Market capitalization of listed domestic companies'},
 {'Indicator Name': 'Access to electricity (% of population)',
  'indicator_new': 'Access to electricity'},
 {'Indicator Name': 'Electric power consumption (kWh per capita)',
  'indicator_new': 'Electric power consumption'},
 {'Indicator Name': 'Birth rate, crude (per 1,000 people)',
  'indicator_new': 'Birth rate'},
 {'Indicator Name': 'Death rate, crude (per 1,000 people)',
  'indicator_new': 'Death rate'},
 {'Indicator Name': 'Human capital index (HCI) (scale 0-1)',
  'indicator_new': 'HCI'},
 {'Indicator Name': 'Life expectancy at birth, total (years)',
  'indicator_new': 'Life expectancy at birth, total (years)'},
 {'Indicator Name': 'Population, total', 'indicator_new': 'Population'},
 {'Indicator Name': 'Total alcohol consumption pe

# create the compiled data

In [6]:
# merge data to obatin selected items (indicators + countries)

shape1 = data_df.shape
data_selected = pd.merge(data_df, countries_select_df, how='inner', on='Country Name')
shape2 = data_selected.shape
data_selected = pd.merge(data_selected, indicators_df, how='inner', on='Indicator Code')
shape3 = data_selected.shape

# control 

print(f'original shape : {shape1}')
print(f'merge countries : {shape2}')
print(f'merge indicators : {shape3}')
print(f'number of countries : {len(data_selected['Country Code_x'].unique())}')
print(f'number of indicators : {len(data_selected['Indicator Code'].unique())}')

original shape : (397936, 68)
merge countries : (74800, 69)
merge indicators : (800, 72)
number of countries : 50
number of indicators : 16


In [7]:
data_selected.columns

Index(['Country Name', 'Country Code_x', 'Indicator Name_x', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
       '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
       '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
       '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023', 'Country Code_y', 'Indicator Name_y', 'indicator_new',
       'Category'],
      dtype='object')

In [8]:
data_selected = data_selected.drop(columns=['Country Code_x', 'Country Code_y','Indicator Name_x', 'Indicator Code', 'Indicator Name_y', 'Category'])


In [9]:

data_selected = data_selected.rename(columns={
                                        'Country Name' : 'country'    ,
                                        'indicator_new' : 'Indicator' , 

                                        })

data_selected.columns

Index(['country', '1960', '1961', '1962', '1963', '1964', '1965', '1966',
       '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975',
       '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984',
       '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993',
       '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002',
       '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011',
       '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020',
       '2021', '2022', '2023', 'Indicator'],
      dtype='object')

In [10]:
# unpivot and get the years as a column

data_selected_long = pd.melt(
                    data_selected, 
                    id_vars=['country', 'Indicator'],
                    value_vars=['1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
                                '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
                                '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
                                '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
                                '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
                                '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
                                '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
                                '2023'],
                     var_name='year' ,           
                     value_name='Value' )


data_selected_long.head()

,country,Indicator,year,Value
0,Algeria,Access to electricity,1960,NaN
1,Algeria,Arable land,1960,NaN
2,Algeria,Armed forces personnel,1960,NaN
3,Algeria,Birth rate,1960,48.722
4,Algeria,Death rate,1960,23.785


# race bar chart : method matplotlib

### prepare data for race bar chart

In [11]:
data_long_new = data_selected_long.copy()

In [12]:
# create date from year (we need format YYYY-MM-DD)

data_long_new['date'] = pd.to_datetime(data_long_new[['year']].assign(month= 1, day=1))

data_long_new.head(5)

,country,Indicator,year,Value,date
0,Algeria,Access to electricity,1960,NaN,1960-01-01
1,Algeria,Arable land,1960,NaN,1960-01-01
2,Algeria,Armed forces personnel,1960,NaN,1960-01-01
3,Algeria,Birth rate,1960,48.722,1960-01-01
4,Algeria,Death rate,1960,23.785,1960-01-01


### launch the loop to generate the race charts as GIF files

In [18]:
# loop through the indicators lists generated above

for indicator in indicators_list : 

    # filter the DF to select the rows related to the indicator
    data_selected_wide = data_long_new[data_long_new['Indicator']==indicator['indicator_new']]

    # pivot the DF to obtain the expected format
    data_selected_wide = data_selected_wide.pivot_table(columns='country', index=['date'], values = 'Value')
    
    # drop NAN values so that the chart will start with the 1st year of available data and ends when there will be no data
    data_selected_wide = data_selected_wide.dropna(how='all')

    # fillNA the other NAN values i between
    data_selected_wide = data_selected_wide.fillna(0)

    # save the 1st and last year : information to be added in the chart
    min_year = data_selected_wide.index.min().year
    max_year = data_selected_wide.index.max().year

    # lauch the chart generation (n_bars = 8 ; steps-per_periode = 6)
    bcr.bar_chart_race(
        df=data_selected_wide, 
        filename=f'race_charts/race_chart_{indicator['indicator_new']}.mp4', 

        orientation='h', 
        sort='desc', 
        n_bars=8, 
        fixed_order=False, 
        fixed_max=True, 
        steps_per_period=20, 
        period_length=750, 
        # end_period_pause=0,
        interpolate_period=False, 
        period_label={'x': .98, 'y': .3, 'ha': 'right', 'va': 'center'}, 

        period_fmt='%Y',
        period_summary_func=lambda v, r: {'x': .98, 'y': .2, 
                                          's': f'Average {indicator['indicator_new']}: {v.mean():,.0f}', 
                                          'ha': 'right', 'size': 11}, 
        perpendicular_bar_func='median', 
        # colors='dark12', 
        title=f'Evolution of {indicator['Indicator Name']} between {min_year} and {max_year}', 
        bar_size=.95, 
        # bar_textposition='inside',
        # bar_texttemplate='{x:,.0f}', 
        # bar_label_font=7, 
        # tick_label_font=7, 
        # tick_template='{x:,.0f}',
        shared_fontdict=None, 
        scale='linear', 
        fig=None, 
        writer=None, 
        bar_kwargs={'alpha': .7},
        figsize = (6, 3.5), 
        dpi = 144,
        filter_column_colors=False) 
    
    print(f'GIF for {indicator["indicator_new"]} is done')
    
    plt.close('all')  # ✅ Releases memory after saving each GIF

c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:889: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_values.iloc[:, 0] = df_values.iloc[:, 0].fillna(method='ffill')
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:286: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(self.df_values.columns)
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:287: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([max_val] * len(ax.get_xticks()))
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:226: UserWarning: Some of your columns never make an appearance in the animation. To reduce color repetition, set `filter_column_colors

GIF for Arable land is done


c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:889: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_values.iloc[:, 0] = df_values.iloc[:, 0].fillna(method='ffill')
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:286: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(self.df_values.columns)
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:287: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([max_val] * len(ax.get_xticks()))


GIF for Market capitalization of listed domestic companies is done


c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:889: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_values.iloc[:, 0] = df_values.iloc[:, 0].fillna(method='ffill')
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:286: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(self.df_values.columns)
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:287: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([max_val] * len(ax.get_xticks()))
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:226: UserWarning: Some of your columns never make an appearance in the animation. To reduce color repetition, set `filter_column_colors

...

GIF for GDP per capita is done


c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:889: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_values.iloc[:, 0] = df_values.iloc[:, 0].fillna(method='ffill')
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:286: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(self.df_values.columns)
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:287: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([max_val] * len(ax.get_xticks()))
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:226: UserWarning: Some of your columns never make an appearance in the animation. To reduce color repetition, set `filter_column_colors

GIF for GNI is done


c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:889: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_values.iloc[:, 0] = df_values.iloc[:, 0].fillna(method='ffill')
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:286: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(self.df_values.columns)
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:287: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([max_val] * len(ax.get_xticks()))
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:226: UserWarning: Some of your columns never make an appearance in the animation. To reduce color repetition, set `filter_column_colors

GIF for Debt service is done


## test with 1 indicator
to adjust or modify the code

In [13]:
indicator_test = 'GDP'
indicator_plus_test = 'GDP (constant 2015 US$)'

data_selected_wide = data_long_new[data_long_new['Indicator']==indicator_test]

In [14]:

data_selected_wide = data_selected_wide.pivot_table(columns='country', index=['date'], values = 'Value')
data_selected_wide = data_selected_wide.dropna(how='all')
data_selected_wide = data_selected_wide.fillna(0)

data_selected_wide.head()

country,Algeria,Angola,Benin,Botswana,Burkina Faso,Burundi,Cameroon,Central African Republic,Chad,"Congo, Dem. Rep.",...,Somalia,South Africa,South Sudan,Sudan,Tanzania,Togo,Tunisia,Uganda,Zambia,Zimbabwe
date,,,,,,,,,,,,,,,,,,,,,
1960-01-01,2.728751e+10,0.0,1.651121e+09,2.032976e+08,1.166783e+09,7.985360e+08,4.827329e+09,9.803715e+08,1.696963e+09,1.916802e+10,...,9.170458e+08,6.565279e+10,0.0,9.621210e+09,8.038069e+09,7.517995e+08,0.000000e+00,0.0,3.745698e+09,4.329261e+09
1961-01-01,2.357493e+10,0.0,1.702987e+09,2.161934e+08,1.213967e+09,6.887682e+08,4.884470e+09,1.028935e+09,1.720683e+09,1.708802e+10,...,8.871319e+08,6.817696e+10,0.0,9.623364e+09,8.129566e+09,8.432884e+08,6.137830e+09,0.0,3.796692e+09,4.602704e+09
1962-01-01,1.893419e+10,0.0,1.644636e+09,2.306063e+08,1.288368e+09,7.511923e+08,5.033034e+09,9.907222e+08,1.812913e+09,2.071080e+10,...,9.499112e+08,7.238889e+10,0.0,1.028890e+10,8.869092e+09,8.751112e+08,5.856098e+09,0.0,3.702122e+09,4.668729e+09
1963-01-01,2.543122e+10,0.0,1.722427e+09,2.442606e+08,1.272031e+09,7.822572e+08,5.221397e+09,9.837177e+08,1.783916e+09,2.179066e+10,...,9.765923e+08,7.772663e+10,0.0,9.995979e+09,9.424297e+09,9.188641e+08,5.227505e+09,0.0,3.823270e+09,4.960260e+09
1964-01-01,2.691625e+10,0.0,1.836982e+09,2.609492e+08,1.301070e+09,8.313285e+08,5.406609e+09,1.004182e+09,1.739123e+09,2.125892e+10,...,9.169164e+08,8.389783e+10,0.0,9.883979e+09,9.943685e+09,1.050133e+09,4.983069e+09,0.0,4.290246e+09,4.905391e+09


In [15]:
min_year = data_selected_wide.index.min().year
max_year = data_selected_wide.index.max().year

print(f'min is {min_year}')
print(f'max is {max_year}')

min is 1960
max is 2023


In [17]:
bcr.bar_chart_race(
        df=data_selected_wide, 
        filename=f'test8_bars_dpi144_steps20.mp4', 

        orientation='h', 
        sort='desc', 
        n_bars=8, 
        fixed_order=False, 
        fixed_max=True, 
        steps_per_period=20, 
        period_length=750, 
        # end_period_pause=0,
        interpolate_period=False, 
        period_label={'x': .98, 'y': .3, 'ha': 'right', 'va': 'center'}, 
        # period_fmt='%B %d, %Y',
        period_fmt='%Y',
        period_summary_func=lambda v, r: {'x': .98, 'y': .2, 
                                          's': f'Average {indicator_test}: {v.mean():,.0f}', 
                                          'ha': 'right', 'size': 11}, 
        perpendicular_bar_func='median', 
        # colors='dark12', 
        title=f'Evolution of {indicator_plus_test} from {min_year} to {max_year} ', 
        bar_size=.95, 
        # bar_textposition='inside',
        # bar_texttemplate='{x:,.0f}', 
        # bar_label_font=7, 
        # tick_label_font=7, 
        # tick_template='{x:,.0f}',
        shared_fontdict=None, 
        scale='linear', 
        fig=None, 
        writer=None, 
        bar_kwargs={'alpha': .7},
        figsize = (6, 3.5), 
        dpi = 144,
        filter_column_colors=False
        ) 

c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:889: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_values.iloc[:, 0] = df_values.iloc[:, 0].fillna(method='ffill')
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:286: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(self.df_values.columns)
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:287: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([max_val] * len(ax.get_xticks()))
c:\Users\nazim\anaconda3\Lib\site-packages\bar_chart_race\_make_chart.py:226: UserWarning: Some of your columns never make an appearance in the animation. To reduce color repetition, set `filter_column_colors

# Race bar chart method PLOTLY

In [ ]:
data_new = data_selected_long.copy()

In [ ]:
indicator = 'GDP per capita'
data = data_new[data_new['Indicator']==indicator]
data

In [ ]:
my_raceplot = barplot(
                        data,  
                        item_column='country', 
                        value_column='Value', 
                        time_column='year',
                        top_entries = 12
                        
                        )

my_raceplot.plot(
                item_label = 'Top 10 countries', 
                value_label = f'{indicator}', 
                frame_duration = 800

                )